In [ ]:
"""
QPSK-Quantized CNN on CIFAR-10  — v7 TURBO
============================================
Architecture : IDENTICAL to v7 (hard 16-QAM inputs + QPSK weights/acts)
               Same VGG-style CNN. Same 28M parameters. Same inference path.
               Zero float weights. Fully quantized. Hardware-ready.

What changed : Training loop only. The drone sees the exact same model.

Upgrades vs v7:
  ① Clipped STE on QPSK weights
       Old: gradients flow through sign() everywhere, forever.
       New: gradient is zeroed where |w| > 1.2 (outside the QPSK ±1 range).
            Latent weights that have "run away" stop receiving misleading
            gradient updates that can't change their quantized output anyway.

  ② Knowledge Distillation (Teacher = float VGG, Student = this model)
       Old: student trains against hard one-hot labels  [0,0,1,0,...,0]
       New: student trains against teacher's soft probabilities [0.02,0.01,0.87,...]
            Soft targets carry inter-class similarity information that hard
            labels throw away. The QPSK weights get a much richer gradient
            signal to follow. Typical gain: +2% to +4% on quantized networks.
       Teacher is trained first (float, same architecture, ~20 epochs → ~90%).
       KD loss = α × KL(student||teacher) + (1-α) × CE(student, hard_labels)

  ③ CutMix augmentation
       Pastes a random rectangle from one image onto another, mixes labels
       proportionally by area. Forces the QPSK filters to learn structural
       shape features rather than memorizing exact pixel layouts (which the
       hard input quantization already distorts heavily).
       Typical gain: +1.5% on CIFAR-10.

  ④ AutoAugment (CIFAR-10 policy)
       Learned augmentation policy from Cubuk et al. 2019. ~+1%.

  ⑤ Label smoothing (ε=0.1)
       Prevents overconfident logits. Better calibration and generalization.
       Particularly useful when combined with CutMix.

  ⑥ Cosine LR with linear warmup
       Linear warmup for 10 epochs (stabilizes early quantized training where
       STE approximation error is highest), then cosine decay to near-zero.
       Better than MultiStepLR: smoother decay, less sensitive to milestone tuning.

  ⑦ Gradient clipping (max_norm=5.0)
       Quantized networks can produce noisy gradient spikes. Clipping prevents
       a single bad batch from destabilizing latent weights that have just
       committed to a sign.

RESULT:
  Best test accuracy achieved: 86.53%
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import math
import numpy as np

# ─────────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")


# ─────────────────────────────────────────────
# CONSTANTS  (fixed to 16-QAM input, QPSK weights)
# ─────────────────────────────────────────────
INPUT_LEVELS    = 16                    # hard 16-QAM input per RGB channel
QPSK_LEVEL      = 1.0 / math.sqrt(2)   # ±1/√2 ≈ ±0.7071
CLIP_THRESHOLD  = 1.2                   # clipped STE: zero gradient where |w| > this


# ─────────────────────────────────────────────
# QAM CONSTELLATION TABLE
# ─────────────────────────────────────────────
def get_qam_levels(n_levels: int) -> torch.Tensor:
    # Builds the 1-D axis of a square QAM constellation with n_levels points.
    # Generates n_per_axis odd integers centred at zero, normalises to RMS=1.
    # For 16-QAM yields 4 levels printed at startup for operator verification.
    n_per_axis = int(math.sqrt(n_levels))
    raw = torch.arange(-(n_per_axis - 1), n_per_axis, 2, dtype=torch.float32)
    rms = raw.pow(2).mean().sqrt()
    return raw / rms


# ─────────────────────────────────────────────
# HARD 16-QAM INPUT QUANTIZER  (no STE needed)
# ─────────────────────────────────────────────
class HardQAMInput(nn.Module):
    # Hard 16-QAM quantization of normalized RGB pixels.
    # Zero learnable parameters; no STE needed since image tensors have
    # requires_grad=False so the argmin snap never blocks any gradient path.
    def __init__(self, n_levels: int = 16):
        # Registers constellation levels as a non-trainable buffer so they
        # move to the correct device with the model. Caches max_val for the
        # Tanh bounding scale used in the forward pass.
        super().__init__()
        levels  = get_qam_levels(n_levels)
        self.register_buffer('levels', levels)
        self.max_val    = levels.abs().max().item()
        self.n_per_axis = int(math.sqrt(n_levels))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Tanh-bounds input to [-max_level, +max_level], then argmin-snaps
        # each element to the nearest constellation point. No STE wrapper
        # needed since pixel values carry no gradient.
        x = torch.tanh(x) * self.max_val
        dists = (x.unsqueeze(-1) - self.levels.view(1, 1, 1, 1, -1)).abs()
        return self.levels[dists.argmin(dim=-1)]

    @torch.no_grad()
    def verify(self, x: torch.Tensor):
        # Runs forward in no-grad mode and prints sanity-check statistics:
        # unique level count (should equal n_per_axis), exact level values,
        # and output range. Called once before training starts.
        xq = self.forward(x)
        u  = xq.unique()
        ok = "✓ OK" if u.numel() == self.n_per_axis else "✗ BUG"
        print(f"  Unique levels : {u.numel()} (expected {self.n_per_axis})  {ok}")
        print(f"  Level values  : {[f'{v:.4f}' for v in u.tolist()]}")
        print(f"  Output range  : [{xq.min():.4f}, {xq.max():.4f}]")


# ─────────────────────────────────────────────
# QPSK WEIGHT/ACTIVATION QUANTIZER  — CLIPPED STE
# ─────────────────────────────────────────────
class QPSKQuantizeClipped(torch.autograd.Function):
    # Snaps each element to ±1/√2 (QPSK) in the forward pass with a CLIPPED
    # STE in the backward pass: gradient is zeroed where |w| > CLIP_THRESHOLD.
    # Runaway latent weights (e.g. w=+3.5) can't change their quantized output
    # until they cross zero, so sending gradients there is misleading — clipping
    # focuses updates on weights near the ±1 decision boundary.
    @staticmethod
    def forward(ctx, x):
        # Snaps to ±1/√2 and saves input so the backward pass can build the
        # clipping mask without recomputing the forward.
        ctx.save_for_backward(x)
        return torch.sign(x) / math.sqrt(2)

    @staticmethod
    def backward(ctx, grad_output):
        # Passes gradient unchanged where |x| ≤ CLIP_THRESHOLD, zeros it
        # beyond — prevents runaway latent weights from absorbing gradient
        # energy that cannot influence their quantized output.
        (x,) = ctx.saved_tensors
        mask = (x.abs() <= CLIP_THRESHOLD).to(grad_output.dtype)
        return grad_output * mask

qpsk_quantize = QPSKQuantizeClipped.apply


# ─────────────────────────────────────────────
# DIAGNOSTICS HELPERS
# ─────────────────────────────────────────────
def sign_commitment(w: torch.Tensor) -> float:
    # Fraction of latent weights with |w| > 0.1 — proxy for how firmly each
    # weight has committed to a QPSK sign. Close to 1.0 means stable snapping.
    return (w.abs() > 0.1).float().mean().item()

def qpsk_snap_error(w_real: torch.Tensor, w_imag: torch.Tensor) -> float:
    # Mean absolute distance between latent weights and their hard-snapped
    # QPSK counterparts, averaged over real and imaginary components.
    # Small values indicate convergence close to the discrete QPSK grid.
    q_r = torch.sign(w_real) / math.sqrt(2)
    q_i = torch.sign(w_imag) / math.sqrt(2)
    return ((w_real - q_r).abs().mean() + (w_imag - q_i).abs().mean()).item() / 2.0

def sign_penalty(w_real: torch.Tensor, w_imag: torch.Tensor) -> torch.Tensor:
    # Bimodal regularizer: exp(-|w|) penalizes weights near zero, pushing
    # them toward a committed QPSK sign. Peaks at zero regardless of
    # constellation width so identical in form to v7's penalty.
    return (torch.exp(-w_real.abs()) + torch.exp(-w_imag.abs())).mean()


# ─────────────────────────────────────────────
# QPSK CONV LAYER
# ─────────────────────────────────────────────
class QPSKConv2d(nn.Module):
    # Drop-in Conv2d with QPSK-constrained latent weights. Stores separate
    # real/imag tensors, snaps both via clipped STE, runs two convolutions,
    # and returns complex magnitude sqrt(r²+i²). Identical to v7's QPSKConv2d
    # except qpsk_quantize now uses the clipped STE instead of unclipped.
    def __init__(self, in_channels, out_channels, kernel_size,
                 padding=0, quantize_input=True):
        # Allocates real/imag latent weight tensors, stores padding and the
        # quantize_input flag, then calls _init_weights.
        super().__init__()
        self.quantize_input = quantize_input
        self.w_real = nn.Parameter(
            torch.Tensor(out_channels, in_channels, kernel_size, kernel_size))
        self.w_imag = nn.Parameter(
            torch.Tensor(out_channels, in_channels, kernel_size, kernel_size))
        self.padding = padding
        self._init_weights()

    def _init_weights(self):
        # Kaiming uniform scaled by 0.3 so initial weights sit near the QPSK
        # boundary, giving sign-penalty room to push toward commitment without
        # saturating. Keeps the clipped-STE mask fully open early in training.
        nn.init.kaiming_uniform_(self.w_real, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_imag, a=math.sqrt(5))
        with torch.no_grad():
            self.w_real.data *= 0.3
            self.w_imag.data *= 0.3

    def forward(self, x):
        # Optionally re-quantizes inputs to QPSK via clipped STE, then runs
        # two convolutions with snapped real/imag weights and returns the
        # complex magnitude. First layer sets quantize_input=False so hard
        # 16-QAM pixel values pass through unmodified.
        if self.quantize_input:
            x = qpsk_quantize(x)
        w_q_real = qpsk_quantize(self.w_real)
        w_q_imag = qpsk_quantize(self.w_imag)
        out_real = F.conv2d(x, w_q_real, padding=self.padding)
        out_imag = F.conv2d(x, w_q_imag, padding=self.padding)
        return torch.sqrt(out_real ** 2 + out_imag ** 2 + 1e-8)

    def diagnostics(self):
        # Returns sign_commitment and snap_error for real and imag weights.
        # Used in the periodic diagnostic pass to monitor QPSK grid convergence.
        return {
            "sign_commit_real": sign_commitment(self.w_real),
            "sign_commit_imag": sign_commitment(self.w_imag),
            "snap_error":       qpsk_snap_error(self.w_real, self.w_imag),
        }

    def penalty(self):
        # Returns the bimodal sign-penalty for this layer's weights,
        # to be scaled by λ_conv and summed into the total training loss.
        return sign_penalty(self.w_real, self.w_imag)


# ─────────────────────────────────────────────
# QPSK LINEAR LAYER
# ─────────────────────────────────────────────
class QPSKLinear(nn.Module):
    # FC analogue of QPSKConv2d: real/imag latent weights quantized to QPSK
    # at every forward pass via clipped STE, returning complex magnitude of
    # two linear projections. Identical to v7's QPSKLinear except for the
    # clipped STE.
    def __init__(self, in_features, out_features):
        # Allocates real/imag latent weight matrices and calls _init_weights.
        super().__init__()
        self.w_real = nn.Parameter(torch.Tensor(out_features, in_features))
        self.w_imag = nn.Parameter(torch.Tensor(out_features, in_features))
        self._init_weights()

    def _init_weights(self):
        # Kaiming uniform scaled by 0.3 — same as QPSKConv2d. Places weights
        # near the QPSK boundary so sign-penalty is effective from epoch 1
        # and the clipped-STE mask starts fully open.
        nn.init.kaiming_uniform_(self.w_real, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_imag, a=math.sqrt(5))
        with torch.no_grad():
            self.w_real.data *= 0.3
            self.w_imag.data *= 0.3

    def forward(self, x):
        # Quantizes input activations to QPSK, applies two linear projections
        # with snapped real/imag weights, and returns element-wise complex
        # magnitude of the results.
        x = qpsk_quantize(x)
        w_q_real = qpsk_quantize(self.w_real)
        w_q_imag = qpsk_quantize(self.w_imag)
        out_real = F.linear(x, w_q_real)
        out_imag = F.linear(x, w_q_imag)
        return torch.sqrt(out_real ** 2 + out_imag ** 2 + 1e-8)

    def diagnostics(self):
        # Same health metrics as QPSKConv2d.diagnostics() so all QPSK layers
        # share a uniform monitoring interface during diagnostic logging.
        return {
            "sign_commit_real": sign_commitment(self.w_real),
            "sign_commit_imag": sign_commitment(self.w_imag),
            "snap_error":       qpsk_snap_error(self.w_real, self.w_imag),
        }

    def penalty(self):
        # Returns the bimodal sign-penalty for this layer's weights,
        # to be scaled by λ_fc and summed into the total training loss.
        return sign_penalty(self.w_real, self.w_imag)


# ─────────────────────────────────────────────
# STUDENT NETWORK  (identical to v7 16-QAM)
# ─────────────────────────────────────────────
class QPSKNet(nn.Module):
    # VGG-style student: hard 16-QAM inputs, QPSK weights/activations, float
    # linear head. Architecture is byte-for-byte identical to v7 (INPUT_LEVELS=16);
    # only the clipped STE replaces the unclipped one — same checkpoint format
    # and inference path, turbo upgrades affect training only.
    def __init__(self, num_classes=10):
        # Builds hard 16-QAM input quantizer, six QPSKConv2d layers with BN
        # and three MaxPool2d downsamples, two QPSKLinear layers with BN1d,
        # and a float nn.Linear classification head.
        super().__init__()
        self.input_quantizer = HardQAMInput(INPUT_LEVELS)

        self.conv1 = QPSKConv2d(3,   128, 3, padding=1, quantize_input=False)
        self.bn1   = nn.BatchNorm2d(128, momentum=0.1, eps=1e-4)
        self.conv2 = QPSKConv2d(128, 128, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(128, momentum=0.1, eps=1e-4)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv3 = QPSKConv2d(128, 256, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(256, momentum=0.1, eps=1e-4)
        self.conv4 = QPSKConv2d(256, 256, 3, padding=1)
        self.bn4   = nn.BatchNorm2d(256, momentum=0.1, eps=1e-4)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv5 = QPSKConv2d(256, 512, 3, padding=1)
        self.bn5   = nn.BatchNorm2d(512, momentum=0.1, eps=1e-4)
        self.conv6 = QPSKConv2d(512, 512, 3, padding=1)
        self.bn6   = nn.BatchNorm2d(512, momentum=0.1, eps=1e-4)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.fc1    = QPSKLinear(8192, 1024)
        self.bn_fc1 = nn.BatchNorm1d(1024, momentum=0.1, eps=1e-4)
        self.fc2    = QPSKLinear(1024, 1024)
        self.bn_fc2 = nn.BatchNorm1d(1024, momentum=0.1, eps=1e-4)
        self.fc_out = nn.Linear(1024, num_classes)

    def forward(self, x):
        # Hard 16-QAM input → three QPSK conv blocks (BN + MaxPool) →
        # flatten → two QPSK FC layers (BN) → float linear head.
        # Single-tensor call signature — no β, no STE residual, no teacher.
        x = self.input_quantizer(x)
        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = self.pool1(x)
        x = self.bn3(self.conv3(x))
        x = self.bn4(self.conv4(x))
        x = self.pool2(x)
        x = self.bn5(self.conv5(x))
        x = self.bn6(self.conv6(x))
        x = self.pool3(x)
        x = x.view(x.size(0), -1)
        x = self.bn_fc1(self.fc1(x))
        x = self.bn_fc2(self.fc2(x))
        return self.fc_out(x)

    def qpsk_layers(self):
        # Returns (name, layer) pairs for all six conv and both FC layers —
        # single iteration point for diagnostics, penalty, and weight clipping.
        return [
            ("conv1", self.conv1), ("conv2", self.conv2),
            ("conv3", self.conv3), ("conv4", self.conv4),
            ("conv5", self.conv5), ("conv6", self.conv6),
            ("fc1",   self.fc1),   ("fc2",   self.fc2),
        ]

    def get_all_diagnostics(self):
        # Collects diagnostics from every QPSK layer in one call, keyed by
        # layer name. Used in the periodic logging pass without touching
        # individual layers directly.
        return {name: layer.diagnostics() for name, layer in self.qpsk_layers()}

    def total_sign_penalty(self, lambda_conv, lambda_fc):
        # Sums each layer's sign-penalty weighted by λ_conv or λ_fc.
        # Resulting scalar is added to the combined CE + KD loss every step.
        p = torch.tensor(0.0, device=next(self.parameters()).device)
        for name, layer in self.qpsk_layers():
            lam = lambda_fc if name.startswith("fc") else lambda_conv
            p = p + lam * layer.penalty()
        return p


# ─────────────────────────────────────────────
# TEACHER NETWORK  (float VGG, same topology)
# Used only during training — discarded afterward.
# The drone never sees this model.
# ─────────────────────────────────────────────
class FloatVGGTeacher(nn.Module):
    # Float VGG-style teacher with identical channel widths to the student.
    # Trained first to ~90%+, then frozen to provide soft KD targets.
    # Same topology maximises relevance of soft labels for the student.
    # Never broadcast over OFDM — discarded after student training completes.
    def __init__(self, num_classes=10):
        # Three pairs of float Conv2d→BN→ReLU blocks with MaxPool2d, followed
        # by three float linear layers with ReLU and 0.3 Dropout. The block()
        # helper keeps the constructor concise.
        super().__init__()
        def block(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1, bias=False),
                nn.BatchNorm2d(oc),
                nn.ReLU(inplace=True),
            )
        self.features = nn.Sequential(
            block(3,   128), block(128, 128), nn.MaxPool2d(2, 2),
            block(128, 256), block(256, 256), nn.MaxPool2d(2, 2),
            block(256, 512), block(512, 512), nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(8192, 1024), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(1024, 1024), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        # Runs input through the float conv extractor, flattens, and passes
        # through the float classifier. No quantization applied anywhere.
        return self.classifier(self.features(x).view(x.size(0), -1))


# ─────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────
def get_cifar10_loaders(batch_size=128, num_workers=2):
    # CIFAR-10 loaders with AutoAugment (torchvision ≥ 0.12) or ColorJitter
    # fallback, plus random crop and horizontal flip. CutMix is applied in
    # the training loop, not here. num_workers forced to 0 on MPS; pin_memory
    # enabled only for CUDA.
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2023, 0.1994, 0.2010)

    try:
        aa_policy = transforms.AutoAugmentPolicy.CIFAR10
        train_transform = transforms.Compose([
            transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
            transforms.RandomHorizontalFlip(),
            transforms.AutoAugment(aa_policy),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        print("  Augmentation : AutoAugment(CIFAR10) + CutMix in loop")
    except AttributeError:
        # torchvision < 0.12 fallback
        train_transform = transforms.Compose([
            transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        print("  Augmentation : Basic fallback + CutMix in loop")

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    safe_workers = 0 if device.type == "mps" else num_workers
    kw = dict(num_workers=safe_workers, pin_memory=(device.type == "cuda"))

    train_set = torchvision.datasets.CIFAR10('./data', True,  train_transform, download=True)
    test_set  = torchvision.datasets.CIFAR10('./data', False, test_transform,  download=True)

    return (
        torch.utils.data.DataLoader(train_set, batch_size, shuffle=True,  **kw),
        torch.utils.data.DataLoader(test_set,  batch_size, shuffle=False, **kw),
    )


# ─────────────────────────────────────────────
# CUTMIX
# ─────────────────────────────────────────────
def cutmix(x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0):
    # Samples λ ~ Beta(alpha, alpha), pastes a random rectangle from a shuffled
    # batch copy onto the originals, and returns mixed images, ya, yb, and
    # lam_actual (true area fraction kept from ya after integer rounding).
    # Caller mixes CE as: lam_actual·CE(ya) + (1−lam_actual)·CE(yb).
    # Applied only after warmup; teacher always sees clean (un-mixed) images.
    lam  = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape

    ch = int(H * math.sqrt(1.0 - lam))
    cw = int(W * math.sqrt(1.0 - lam))
    cy = np.random.randint(H)
    cx = np.random.randint(W)
    y1, y2 = max(0, cy - ch // 2), min(H, cy + ch // 2)
    x1, x2 = max(0, cx - cw // 2), min(W, cx + cw // 2)

    x_mix = x.clone()
    x_mix[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    lam_actual = 1.0 - (y2 - y1) * (x2 - x1) / (H * W)

    return x_mix, y, y[perm], lam_actual


# ─────────────────────────────────────────────
# LOSS FUNCTIONS
# ─────────────────────────────────────────────
def label_smoothed_ce(outputs, ya, yb=None, lam=1.0,
                      smoothing=0.1, nc=10):
    # Label-smoothed CE with optional CutMix blending. Converts hard labels
    # to soft targets (correct class = 1−ε, others share ε). With yb supplied:
    # loss = lam·NLL(ya) + (1−lam)·NLL(yb). Smoothing prevents overconfident
    # logits on ambiguous CutMix images.
    log_p = F.log_softmax(outputs, dim=1)

    def nll(targets):
        with torch.no_grad():
            soft = torch.full_like(log_p, smoothing / (nc - 1))
            soft.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)
        return -(soft * log_p).sum(dim=1).mean()

    if yb is None:
        return nll(ya)
    return lam * nll(ya) + (1.0 - lam) * nll(yb)


def kd_loss(student_logits, teacher_logits, temperature=4.0):
    # KL divergence between student and teacher soft distributions at
    # temperature T. Higher T flattens the teacher, revealing inter-class
    # similarity hard labels discard. Scaled by T² to keep gradient magnitude
    # consistent across temperature choices.
    s_log  = F.log_softmax(student_logits / temperature, dim=1)
    t_soft = F.softmax(teacher_logits  / temperature, dim=1)
    return F.kl_div(s_log, t_soft, reduction='batchmean') * (temperature ** 2)


# ─────────────────────────────────────────────
# SCHEDULER
# ─────────────────────────────────────────────
def make_cosine_scheduler(optimizer, warmup_epochs, total_epochs):
    # Linear warmup for warmup_epochs (stabilises early STE training), then
    # half-cosine decay to zero. Smoother than MultiStepLR and less sensitive
    # to milestone tuning. Used by both teacher and student.
    def lr_fn(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        t = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * t))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)


# ─────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────
def evaluate(model, loader):
    # Runs inference over a full DataLoader in eval mode (BN uses running
    # stats, Dropout off) with gradients disabled. Returns top-1 accuracy %.
    # Works for both float teacher and quantized student via the same interface.
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            _, pred = model(images).max(1)
            correct += pred.eq(labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total


# ─────────────────────────────────────────────
# TEACHER TRAINING
# ─────────────────────────────────────────────
def train_teacher(train_loader, test_loader, epochs=25):
    # Trains float teacher with AdamW + cosine LR + label-smoothed CE +
    # CutMix (50% per batch) + gradient clipping. Best checkpoint reloaded
    # before return so student always distils from the highest-accuracy
    # snapshot. Teacher is discarded after student training completes.
    print("\n" + "=" * 60)
    print("  PHASE 1: Training Float Teacher")
    print(f"  Target: ~90%+  |  Epochs: {epochs}")
    print("=" * 60)

    teacher = FloatVGGTeacher().to(device)
    opt = torch.optim.AdamW(teacher.parameters(), lr=3e-3, weight_decay=1e-4)
    sch = make_cosine_scheduler(opt, warmup_epochs=3, total_epochs=epochs)

    best_acc  = 0.0
    best_path = "teacher_v7turbo_best.pth"

    for epoch in range(1, epochs + 1):
        teacher.train()
        correct = total = running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            if np.random.random() < 0.5:
                images, la, lb, lam = cutmix(images, labels)
                out  = teacher(images)
                loss = label_smoothed_ce(out, la, lb, lam)
            else:
                out  = teacher(images)
                loss = label_smoothed_ce(out, labels)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(teacher.parameters(), 5.0)
            opt.step()

            running_loss += loss.item()
            _, pred = out.max(1)
            correct += pred.eq(labels).sum().item()
            total   += labels.size(0)

        sch.step()
        test_acc = evaluate(teacher, test_loader)

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(teacher.state_dict(), best_path)

        lr_now = opt.param_groups[0]['lr']
        print(f"  Teacher Ep {epoch:3d}/{epochs} | "
              f"Loss {running_loss/len(train_loader):.4f} | "
              f"Train {100.0*correct/total:.2f}% | "
              f"Test {test_acc:.2f}% | "
              f"Best {best_acc:.2f}% | "
              f"LR {lr_now:.2e}")

    # Load best teacher weights before returning so the student always
    # distils from the highest-accuracy snapshot rather than the last epoch.
    teacher.load_state_dict(torch.load(best_path, map_location=device))
    teacher.eval()
    print(f"\n  Teacher ready. Best test accuracy: {best_acc:.2f}%")
    print("  Teacher will be discarded after student training.\n")
    return teacher


# ─────────────────────────────────────────────
# STUDENT TRAINING
# ─────────────────────────────────────────────
def train_student_epoch(student, teacher, loader, optimizer,
                        lambda_conv, lambda_fc,
                        kd_alpha, kd_temp, cutmix_prob,
                        epoch, warmup_epochs,
                        log_grads=False):
    # One epoch of student training: CutMix (after warmup) → label-smoothed
    # CE on mixed images + KD loss vs teacher on clean images + sign penalty.
    # Combined loss = (1−α)·CE + α·KD + penalty. Gradient clipping at 5.0,
    # then BinaryConnect weight clipping to [−1, 1] after each optimizer step.
    # Returns epoch-averaged CE, KD, penalty losses and train accuracy.
    student.train()
    teacher.eval()

    running_ce = running_kd = running_pen = 0.0
    correct = total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        # CutMix: activate after warmup for training stability
        use_cutmix = (epoch > warmup_epochs) and (np.random.random() < cutmix_prob)
        if use_cutmix:
            images_mix, la, lb, lam = cutmix(images, labels)
        else:
            images_mix, la, lb, lam = images, labels, labels, 1.0

        # Forward: student on mixed images
        student_logits = student(images_mix)

        # CE loss with label smoothing (+ CutMix mixing if active)
        if use_cutmix:
            ce = label_smoothed_ce(student_logits, la, lb, lam)
        else:
            ce = label_smoothed_ce(student_logits, labels)

        # KD loss: student vs teacher on ORIGINAL (un-mixed) images.
        # Teacher sees clean images so mixing would distort its soft labels.
        with torch.no_grad():
            teacher_logits = teacher(images)
        kd = kd_loss(student(images), teacher_logits, kd_temp)

        # Sign penalty
        pen = student.total_sign_penalty(lambda_conv, lambda_fc)

        # Combined loss
        loss = (1.0 - kd_alpha) * ce + kd_alpha * kd + pen

        optimizer.zero_grad()
        loss.backward()

        # Gradient check on epoch 1, batch 0: confirms clipped STE delivers
        # non-zero gradients to every layer. "NO GRADIENT" = broken path.
        if batch_idx == 0 and log_grads:
            print("\n  [GRADIENT CHECK — Epoch 1, Batch 0]")
            for lname, layer in [("conv1", student.conv1),
                                  ("conv5", student.conv5),
                                  ("fc1",   student.fc1),
                                  ("fc2",   student.fc2)]:
                gr = layer.w_real.grad
                gi = layer.w_imag.grad
                if gr is not None:
                    print(f"  {lname}: grad_real={gr.abs().mean():.6f}  "
                          f"grad_imag={gi.abs().mean():.6f}")
                else:
                    print(f"  {lname}: NO GRADIENT ← broken!")
            print()

        # Gradient clipping: prevents a single noisy batch from destabilising
        # latent weights that have just committed to a sign.
        torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=5.0)
        optimizer.step()

        # BinaryConnect weight clipping: keeps latent weights inside [−1, 1]
        # so the CLIP_THRESHOLD of 1.2 always exceeds the max latent magnitude,
        # preventing the gradient mask from permanently zeroing any weight.
        with torch.no_grad():
            for _, layer in student.qpsk_layers():
                layer.w_real.clamp_(-1.0, 1.0)
                layer.w_imag.clamp_(-1.0, 1.0)

        running_ce  += ce.item()
        running_kd  += kd.item()
        running_pen += pen.item()
        _, pred = student_logits.max(1)
        correct += pred.eq(labels).sum().item()
        total   += labels.size(0)

    n = len(loader)
    return (running_ce / n, running_kd / n, running_pen / n,
            100.0 * correct / total)


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    # Two-phase pipeline: Phase 1 trains float teacher to ~90%+; Phase 2
    # trains quantized student with KD + CutMix + clipped STE + cosine LR.
    # Hyperparameters are declared here. Input quantizer is verified before
    # training. Best student checkpoint saved; diagnostic stats printed every
    # DIAG_EVERY epochs. Final summary reports improvement vs v7 baseline.
    # ── Hyperparameters ─────────────────────────────────────────────────────
    # Teacher training
    TEACHER_EPOCHS  = 25       # float teacher: train to ~90%+

    # Student training
    STUDENT_EPOCHS  = 200      # more epochs — cosine LR benefits from longer runs
    BATCH_SIZE      = 128
    LR              = 3e-3     # AdamW peak LR (higher than Adam for cosine schedule)
    WEIGHT_DECAY    = 1e-4
    WARMUP_EPOCHS   = 10       # linear warmup — stabilizes early STE training
    CUTMIX_PROB     = 0.5      # probability of CutMix per batch (after warmup)
    KD_ALPHA        = 0.7      # weight of KD loss vs CE loss (0=pure CE, 1=pure KD)
    KD_TEMP         = 4.0      # temperature for soft label distillation
    LAMBDA_CONV     = 1e-4     # sign penalty for conv layers
    LAMBDA_FC       = 3e-4     # sign penalty for FC layers
    DIAG_EVERY      = 25       # diagnostic print interval

    print("=" * 70)
    print("  QPSK-Quantized VGG  [v7 TURBO — 16-QAM Input + QPSK Weights]")
    print("  Architecture : IDENTICAL to v7 (100% quantized, hardware-ready)")
    print("  Training     : KD + CutMix + AutoAugment + Clipped STE + Warmup")
    print(f"  16-QAM levels: {get_qam_levels(16).tolist()}")
    print(f"  QPSK level   : ±{QPSK_LEVEL:.4f}")
    print(f"  Clipped STE  : |w| ≤ {CLIP_THRESHOLD} (zero gradient beyond)")
    print(f"  KD alpha     : {KD_ALPHA}  |  Temperature: {KD_TEMP}")
    print(f"  CutMix prob  : {CUTMIX_PROB}  (after {WARMUP_EPOCHS} warmup epochs)")
    print(f"  Baseline     : v7 hard 16-QAM = 75.63%")
    print(f"  Target       : 80%+  |  Stretch: 82-83%")
    print("=" * 70)

    train_loader, test_loader = get_cifar10_loaders(BATCH_SIZE)

    # ── Phase 1: Train teacher ──────────────────────────────────────────────
    teacher = train_teacher(train_loader, test_loader, epochs=TEACHER_EPOCHS)

    # ── Phase 2: Train student ──────────────────────────────────────────────
    print("=" * 60)
    print("  PHASE 2: Training Quantized Student (v7 architecture)")
    print(f"  Epochs: {STUDENT_EPOCHS}  |  LR: {LR}  |  Warmup: {WARMUP_EPOCHS}")
    print("=" * 60)

    student = QPSKNet(num_classes=10).to(device)

    total_params = sum(p.numel() for p in student.parameters())
    print(f"\nStudent parameters : {total_params:,} (all quantized at inference)")

    # Input quantizer sanity check: verify the hard 16-QAM snapper produces
    # exactly 4 distinct levels per channel before any training begins.
    sample_imgs = next(iter(test_loader))[0][:64].to(device)
    print("\n[Input Quantizer Check]")
    student.input_quantizer.verify(sample_imgs)
    print()

    optimizer = torch.optim.AdamW(
        student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = make_cosine_scheduler(optimizer, WARMUP_EPOCHS, STUDENT_EPOCHS)

    best_acc  = 0.0
    save_name = "qpsk_cifar10_v7turbo_best.pth"

    for epoch in range(1, STUDENT_EPOCHS + 1):
        ce, kd, pen, train_acc = train_student_epoch(
            student, teacher, train_loader, optimizer,
            LAMBDA_CONV, LAMBDA_FC,
            KD_ALPHA, KD_TEMP, CUTMIX_PROB,
            epoch, WARMUP_EPOCHS,
            log_grads=(epoch == 1)
        )
        test_acc = evaluate(student, test_loader)
        scheduler.step()

        is_best = test_acc > best_acc
        if is_best:
            best_acc = test_acc
            torch.save(student.state_dict(), save_name)

        lr_now = optimizer.param_groups[0]['lr']
        mark   = "  ★" if is_best else ""
        print(f"Ep {epoch:3d}/{STUDENT_EPOCHS} | "
              f"CE {ce:.4f}  KD {kd:.4f}  Pen {pen:.4f} | "
              f"Train {train_acc:.2f}%  Test {test_acc:.2f}%  "
              f"Best {best_acc:.2f}% | "
              f"LR {lr_now:.2e}"
              f"{mark}")

        if epoch % DIAG_EVERY == 0:
            print(f"\n  [DIAGNOSTICS — Epoch {epoch}]")
            for lname, d in student.get_all_diagnostics().items():
                print(f"  {lname:6s}: "
                      f"commit_r={d['sign_commit_real']:.3f}  "
                      f"commit_i={d['sign_commit_imag']:.3f}  "
                      f"snap_err={d['snap_error']:.4f}")
            print()

    print(f"\n{'='*70}")
    print(f"Training complete.")
    print(f"Best student accuracy : {best_acc:.2f}%")
    print(f"v7 baseline (16-QAM)  : 75.63%")
    print(f"Improvement           : +{best_acc - 75.63:.2f}%")
    print(f"Target (80%+)         : {'✓ MET' if best_acc >= 80.0 else '✗ not met'}")
    print(f"Stretch (82-83%)      : {'✓ MET' if best_acc >= 82.0 else f'reached {best_acc:.2f}%'}")
    print(f"Model saved           : {save_name}")
    print(f"{'='*70}")


if __name__ == "__main__":
    main()